In [ ]:
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.ppo import PPO, ActorCritic, PPOData
from training.sim import SKILLS, DuckEnv, hardware


In [ ]:
skill = SKILLS["walk"].with_reward_weights(air_time=0.0)   # 滞空这一项的权重改成 0
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()
rewards = env.unwrapped.reward_manager
{name: rewards.get_term_cfg(name).weight for name in ("air_time", "track_linear_velocity", "upright")}

In [ ]:
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")
data = PPOData(env, net)
model = PPO(net)
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
trainer.fit(model, data)

In [ ]:
after = evaluate(net, skill, hw)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
record_video(net, skill, hw, "m05_no_airtime.mp4")
m04 = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim)
m04.load_state_dict(torch.load("../M04-train/m04_walk.pt"))
record_video(m04, SKILLS["walk"], hw, "m04_walk.mp4")
Video("m05_no_airtime.mp4", embed=True)

In [ ]:
torch.save(net.state_dict(), "m05_no_airtime.pt")